In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import plotly.graph_objects as go
import matplotlib.pyplot as plt

## Table of Contents

The data cleaning is divided into sections determining who fits the classification for inclusion in the superager analysis at:
1. [Timepoint 1](#timepoint1) - Participants with all data (age, education, MRI, neuropsych) at tp1 and 2 who fit the classification for inclusion in the superager analysis based off of their neuropsych data from tp1


<a id='timepoint1'></a>
### Timepoint 1

In [ ]:
# Read in clean BBHI data
filtered_df = pd.read_csv("~/Documents/2023:2024/Data/BBHI/Exported data/clean_bbhi.csv")

The following is a calculation based on the criteria proposed by [Sun et al. (2016)](https://pubmed.ncbi.nlm.nih.gov/27629716/) on who can be included in a superager analyses and who is a superager. Our criteria are a slight variation as follows:

- All participants must:
  - Be age 60+
- Control participants must:
  - Score within 1.5 SD of the norm for age and education on the TMT A and B, semantic fluency, digit span forward and backward based on the neuronorma data from [Peña-Casanova et al. (2009a)](https://pubmed.ncbi.nlm.nih.gov/19661109/) and [Peña-Casanova et al. (2009b)](https://pubmed.ncbi.nlm.nih.gov/19648583/) with Spanish adults.
- Superagers must:
  - Score at or above the mean for age 16-29 year olds on the RAVLT long delay free recall based on normative data from [Schmidt (1996)](https://scholar.google.co.uk/scholar?hl=en&as_sdt=0%2C5&q=Schmidt%2C+M.+%281996%29.+Rey+Auditory+and+Verbal+Learning+Test%3A+A+handbook.+Los+Angeles%2C+CA%3A+Western+Psychological+Services&btnG=)
  - Score above 1 SD below the norm for age and education on the TMT B based on the neuronorma data 
  - Score above 1.5 SD below the norm for age and education on the TMT A and B, semantic fluency, digit span forward and backward based on the neuronorma data 

In [ ]:
# Readin the Neuronorma data (from the two publication above) in excel form & put into a df

xls = pd.ExcelFile(
    "/Users/rachelmorse/SuperAgers/Neuronorma data TMT, SDMT, DS, SF.xlsx"
)
score_mappings = {
    sheet_name: pd.read_excel(xls, sheet_name) for sheet_name in xls.sheet_names
}

In [ ]:
# Round down years of education data because some participants have data that is not a whole number (e.g. YoE = 9.5)
# This gives participants the lower education value because they will have their cognitive test scores normalized according to education level, disadvantaging them if it is rounded up

filtered_df["YoE"] = np.floor(filtered_df["YoE"])

To determine who is within 1.5 SD of the norm, first the raw scores from the neuropsychological tests must be transformed into scaled scores based on age and then education. 

In [ ]:
# Create scaled scores for TMT-A that adjust for age

def map_raw_to_scaled(raw_score, age):
    """Maps raw TMT-A scores to scaled TMT-A scores based on age.

    Args:
        raw_score (int): Raw TMT-A score from filtered_df
        age (int): Age of participant
        TMTA (int): Raw TMT-A score from Neuronorma data
        Scale Score (int): Scaled TMT-A score from Neuronorma data

    Returns:
        TMTA_norm_age (int): Scaled TMT-A score for participant based on age
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(
            int, age_range.split("-")
        )  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row["TMTA"])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None


filtered_df["TMTA_norm_age"] = filtered_df.apply(
    lambda row: map_raw_to_scaled(row["w1_tmt_a_raw"], row["age"]), axis=1
)

filtered_df[["id", "age", "w1_tmt_a_raw", "TMTA_norm_age"]].head(10)

In [ ]:
# Create scaled scores for TMT-A that adjust now for education

# Readin Neuronorma data for education and format df

df_edu_data = pd.read_excel(
    "/Users/rachelmorse/SuperAgers/Neuronorma education data.xlsx",
    sheet_name="TMTA",
    index_col=0,
)  # Make sure to specify the sheet name

# Adjust the scaled score calculated above by age to include education
TMTA_norm_list = []

for _, row in filtered_df.iterrows():
    tmta_norm_age = row["TMTA_norm_age"]
    YoE = row["YoE"]

    # If YoE is more than 20, classify it as 20
    if YoE > 20:
        YoE = 20

    TMTA_norm = df_edu_data.loc[tmta_norm_age, YoE]
    TMTA_norm_list.append(TMTA_norm)

filtered_df["TMTA_norm"] = TMTA_norm_list

filtered_df[["id", "YoE", "TMTA_norm_age", "TMTA_norm"]].head(10)

In [ ]:
# Create scaled scores for TMT-B that adjust for age

def map_raw_to_scaled(raw_score, age):
    """Maps raw TMT-B scores to scaled TMT-B scores based on age.

    Args:
        raw_score (int): Raw TMT-B score from filtered_df
        age (int): Age of participant
        TMTB (int): Raw TMT-B score from Neuronorma data
        Scale Score (int): Scaled TMT-B score from Neuronorma data

    Returns:
        TMTB_norm_age (int): Scaled TMT-B score for participant based on age
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(
            int, age_range.split("-")
        )  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row["TMTB"])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None


filtered_df["TMTB_norm_age"] = filtered_df.apply(
    lambda row: map_raw_to_scaled(row["w1_tmt_b_raw"], row["age"]), axis=1
)

filtered_df[["id", "age", "w1_tmt_b_raw", "TMTB_norm_age"]].head(10)

In [ ]:
# Create scaled scores for TMT-B that adjust now for education

# Readin Neuronorma data for education and format df

df_edu_data = pd.read_excel(
    "/Users/rachelmorse/SuperAgers/Neuronorma education data.xlsx",
    sheet_name="TMTB",
    index_col=0,
)  # Make sure to specify the sheet name

# Adjust the scaled score calculated above by age to include education
TMTB_norm_list = []

for index, row in filtered_df.iterrows():
    tmtb_norm_age = row["TMTB_norm_age"]
    YoE = row["YoE"]

    # If YoE is more than 20, classify it as 20
    if YoE > 20:
        YoE = 20

    TMTB_norm = df_edu_data.loc[tmtb_norm_age, YoE]
    TMTB_norm_list.append(TMTB_norm)  # Corrected here

filtered_df["TMTB_norm"] = TMTB_norm_list

filtered_df[["id", "YoE", "TMTB_norm_age", "TMTB_norm"]].head(10)

In [ ]:
# Create scaled scores for Digit Span - forward

# Start by rounding up Digit Span - forward score because some participants have data that is not a whole number (e.g. w1_direct_digits_raw = 4.5)
filtered_df["w1_direct_digits_raw"] = np.ceil(filtered_df["w1_direct_digits_raw"])

In [ ]:
# Create scaled scores for Digit Span - forward that adjust for age

def map_raw_to_scaled(raw_score, age):
    """Maps raw DS-F scores to scaled DS-F scores based on age.

    Args:
        raw_score (int): Raw DS-F score from filtered_df
        age (int): Age of participant
        TMTA (int): Raw DS-F score from Neuronorma data
        Scale Score (int): Scaled DS-F score from Neuronorma data

    Returns:
        dsf_norm_age (int): Scaled DS-F score for participant based on age
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(
            int, age_range.split("-")
        )  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row["DS_F"])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None


filtered_df["dsf_norm_age"] = filtered_df.apply(
    lambda row: map_raw_to_scaled(row["w1_direct_digits_raw"], row["age"]), axis=1
)

filtered_df.sample(10)[["id", "age", "w1_direct_digits_raw", "dsf_norm_age"]].head(10)

In [ ]:
# Create scaled scores for DS-Forward that adjusts now for education

# Readin Neuronorma data for education and format df

df_edu_data = pd.read_excel(
    "/Users/rachelmorse/SuperAgers/Neuronorma education data.xlsx",
    sheet_name="DS_F",
    index_col=0,
)  # Make sure to specify the sheet name

# Adjust the scaled score calculated above by age to include education
dsf_norm_list = []

for index, row in filtered_df.iterrows():
    dsf_norm_age = row["dsf_norm_age"]
    YoE = row["YoE"]

    # If YoE is more than 20, classify it as 20
    if YoE > 20:
        YoE = 20

    dsf_norm = df_edu_data.loc[dsf_norm_age, YoE]
    dsf_norm_list.append(dsf_norm)  # Corrected here

filtered_df["dsf_norm"] = dsf_norm_list

# Display a random 10 rows
filtered_df.sample(10)[["id", "YoE", "dsf_norm_age", "dsf_norm"]]

In [ ]:
# Create scaled scores for Digit Span - backward

# Start by rounding up Digit Span - backward score because some participants have data that is not a whole number (e.g. w1_direct_digits_raw = 2.5)
filtered_df["w1_inverse_digits_raw"] = np.ceil(filtered_df["w1_inverse_digits_raw"])

In [ ]:
# Create scaled scores for Digit Span - backward adjusted for age

def map_raw_to_scaled(raw_score, age):
    """Maps raw DS-B scores to scaled DS-B scores based on age.

    Args:
        raw_score (int): Raw DS-B score from filtered_df
        age (int): Age of participant
        TMTA (int): Raw DS-B score from Neuronorma data
        Scale Score (int): Scaled DS-B score from Neuronorma data

    Returns:
        dsf_norm_age (int): Scaled DS-B score for participant based on age
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(
            int, age_range.split("-")
        )  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row["DS_B"])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None


filtered_df["dsb_norm_age"] = filtered_df.apply(
    lambda row: map_raw_to_scaled(row["w1_inverse_digits_raw"], row["age"]), axis=1
)

filtered_df.sample(10)[["id", "age", "w1_inverse_digits_raw", "dsb_norm_age"]].head(10)

In [ ]:
# Create scaled scores for DS-Backward that adjusts now for education

# Readin Neuronorma data for education and format df

df_edu_data = pd.read_excel(
    "/Users/rachelmorse/SuperAgers/Neuronorma education data.xlsx",
    sheet_name="DS_B",
    index_col=0,
)  # Make sure to specify the sheet name

# Adjust the scaled score calculated above by age to include education
dsb_norm_list = []

for index, row in filtered_df.iterrows():
    dsb_norm_age = row["dsb_norm_age"]
    YoE = row["YoE"]

    # If YoE is more than 20, classify it as 20
    if YoE > 20:
        YoE = 20

    dsb_norm = df_edu_data.loc[dsb_norm_age, YoE]
    dsb_norm_list.append(dsb_norm)  # Corrected here

filtered_df["dsb_norm"] = dsb_norm_list

# Display a random 10 rows
filtered_df.sample(10)[["id", "YoE", "dsb_norm_age", "dsb_norm"]]

In [ ]:
# Create scaled scores for Semantic Fluency that adjust for age

def map_raw_to_scaled(raw_score, age):
    """Maps raw SF scores to scaled SF scores based on age.

    Args:
        raw_score (int): Raw SF score from filtered_df
        age (int): Age of participant
        TMTA (int): Raw SF score from Neuronorma data
        Scale Score (int): Scaled SF score from Neuronorma data

    Returns:
        dsf_norm_age (int): Scaled SF score for participant based on age
    """
    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        min_age, max_age = map(
            int, age_range.split("-")
        )  # Because the sheet names are in the format 'min-max'
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                raw_score_range = str(row["SF"])
                if raw_score_range == "-" or raw_score_range == "—":
                    continue  # Skip this row if there is no data
                elif "≥" in raw_score_range:
                    min_raw_score = int(raw_score_range.replace("≥", ""))
                    if raw_score >= min_raw_score:
                        return row["Scaled Score"]
                elif "≤" in raw_score_range:
                    max_raw_score = int(raw_score_range.replace("≤", ""))
                    if raw_score <= max_raw_score:
                        return row["Scaled Score"]
                elif "–" in raw_score_range:
                    min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
                    if min_raw_score <= raw_score <= max_raw_score:
                        return row["Scaled Score"]
                else:  # raw_score_range is a single value
                    single_value = int(raw_score_range)
                    if raw_score == single_value:
                        return row["Scaled Score"]
    return None


filtered_df["sf_norm_age"] = filtered_df.apply(
    lambda row: map_raw_to_scaled(row["w1_sem_fluency_raw"], row["age"]), axis=1
)

filtered_df.sample(10)[["id", "age", "w1_sem_fluency_raw", "sf_norm_age"]].head(10)

In [ ]:
# Create scaled scores for semantic fluency that adjusts for now for education

# Readin Neuronorma data for education and format df

df_edu_data = pd.read_excel(
    "/Users/rachelmorse/SuperAgers/Neuronorma education data.xlsx",
    sheet_name="SF",
    index_col=0,
)  # Make sure to specify the sheet name

# Adjust the scaled score calculated above by age to include education
sf_norm_list = []

for index, row in filtered_df.iterrows():
    sf_norm_age = row["sf_norm_age"]
    YoE = row["YoE"]

    # If YoE is more than 20, classify it as 20
    if YoE > 20:
        YoE = 20

    sf_norm = df_edu_data.loc[sf_norm_age, YoE]
    sf_norm_list.append(sf_norm)  # Corrected here

filtered_df["sf_norm"] = sf_norm_list

# Display a random 10 rows
filtered_df.sample(10)[["id", "YoE", "sf_norm_age", "sf_norm"]]

In [ ]:
# Calculate who is superager based off of RAVLT score

# Schmidt 1996 - age 16-29 RAVLT-Delayed recall is scoring 12+ (no data adjusted by sex)

filtered_df.loc[filtered_df["w1_delayed_recall_raw"] >= 12, "superager_RAVLT"] = 1
filtered_df.loc[filtered_df["w1_delayed_recall_raw"] < 12, "superager_RAVLT"] = 0

print("Number of superagers by RAVLT criteria:")
print(filtered_df[filtered_df["superager_RAVLT"] == 1]["id"].count())

# Manually check that everything is running correctly
filtered_df[["id", "w1_delayed_recall_raw", "superager_RAVLT"]].head(10)

With the scaled scores calculated, the mean is 10 and the SD is 3 for all variables. For the Neuronorma data, see [Peña-Casanova et al. (2009c)](https://pubmed.ncbi.nlm.nih.gov/19549723/) for more info. 

In [ ]:
# Create a superager variable that = 1 when superagers are above 1SD below the norm for TMT-B and meet the RAVLT criteria

# Define the variables and the lower bound
variables = ["TMTB_norm"]
lower_bound = 10 - 1 * 3  # Scaled score of 10 is the mean and SD is 3 for all variables

# Create new column 'superager' and initialize it to 0
filtered_df["superager"] = 0

# Update 'superager' to 1 for participants who score above the lower bound for TMT-B and have 1 for 'superager_RAVLT'
filtered_df.loc[
    (filtered_df[variables] >= lower_bound).all(axis=1) & (filtered_df["superager_RAVLT"] == 1), 
    "superager"
] = 1

# Display a random sample of 10 rows
sample_df = filtered_df.sample(10)

print(filtered_df[filtered_df["superager"] == 1]["id"].count())

# Display the relevant columns
relevant_columns = ["id", "age", "YoE","superager"]
sample_df[relevant_columns]


In [ ]:
# Create new variables that = 1 when non-superager participants are within 1.5SD of the norm and = 1 when superager participants are above 1.5SD below the norm

variables = ["sf", "dsf", "TMTA", "TMTB", "dsb"]

for var in variables:
    # Calculate lower and upper bounds for each variable
    lower_bound = 10 - 1.5 * 3  # Scaled score of 10 is the mean and SD is 3 for all variables
    upper_bound = 10 + 1.5 * 3

    # Create new column 'normSD_x' where x is the variable
    filtered_df[f"normSD_{var}"] = 0

    # Update 'normSD_x' to 1 if non-superager participants are within the bounds for all the relevant variables 
    filtered_df.loc[
        (filtered_df[f"{var}_norm"] >= lower_bound) 
        & (filtered_df[f"{var}_norm"] <= upper_bound)
        & (filtered_df["superager"] == 0), 
        f"normSD_{var}"
    ] = 1

    # Update 'normSD_x' to 1 if all superager participants meet the lower bound for all the relevant variables 
    filtered_df.loc[
        (filtered_df[f"{var}_norm"] >= lower_bound)
        & (filtered_df["superager"] == 1), 
        f"normSD_{var}"
    ] = 1

# Display a random sample of 10 rows
sample_df = filtered_df.sample(10)

# Display the relevant columns
relevant_columns = ["id", "age", "YoE"] + [f"normSD_{var}" for var in variables]
sample_df[relevant_columns]


In [ ]:
# Calculate who is within the norm for RAVLT

# Define the means and standard deviations for each age group
mean_sd = {
    '60-69': {'mean': 8.8, 'sd': 3.0},  # mean and SD for age group 60-69
    '70+': {'mean': 7.0, 'sd': 2.4}  # mean and SD for age group 70+
}

# Create a new column 'age_group'
filtered_df['age_group'] = pd.cut(filtered_df['age'], bins=[59, 69, np.inf], labels=['60-69', '70+'])

# Create new column 'normSD_var'
filtered_df["normSD_RAVLT"] = 0

# For each age group, update 'normSD_var' based on the bounds for that group
for age_group, params in mean_sd.items():
    lower_bound = params['mean'] - 1.5 * params['sd']
    upper_bound = params['mean'] + 1.5 * params['sd']

    filtered_df.loc[
        (filtered_df['age_group'] == age_group) 
        & (filtered_df['w1_delayed_recall_raw'] >= lower_bound) 
        & (filtered_df['w1_delayed_recall_raw'] <= upper_bound)
        & (filtered_df["superager"] == 0), 
        "normSD_RAVLT"
    ] = 1

    filtered_df.loc[
        (filtered_df['age_group'] == age_group) # because we already know all superagers are above the norm
        & (filtered_df["superager"] == 1),
        "normSD_RAVLT"
    ] = 1

# Display a random sample of 10 rows
sample_df = filtered_df.sample(10)

# Display the relevant columns
relevant_columns = ["id", "age", "YoE", "w1_delayed_recall_raw", "normSD_RAVLT"]
sample_df[relevant_columns]

In [ ]:
# Create a new valiable for those who fit the norm on all the tests and can be included in analysis

filtered_df["norm_neuropsych"] = 0

mask = (
    (filtered_df["normSD_TMTB"] == 1)
    & (filtered_df["normSD_sf"] == 1)
    & (filtered_df["normSD_dsf"] == 1)
    & (filtered_df["normSD_TMTA"] == 1)
    & (filtered_df["normSD_dsb"] == 1)
    & (filtered_df["normSD_RAVLT"] == 1)
)
filtered_df.loc[mask, "norm_neuropsych"] = 1

# Print number of participants within the norm
count_norm_neuropsych = filtered_df[filtered_df["norm_neuropsych"] == 1]["id"].count()
print(f"Number of participants with norm_neuropsych data: {count_norm_neuropsych}")

filtered_df[
    [
        "id",
        "normSD_TMTB",
        "normSD_sf",
        "normSD_dsf",
        "normSD_TMTA",
        "normSD_dsb",
        "normSD_RAVLT",
        "norm_neuropsych",
    ]
]

In [ ]:
# Check whether any superagers do not have norm neuropsych data from the 1.5SD analysis

superger_with_non_normal_neuropsych = filtered_df[
    (filtered_df["superager"] == 1) & (filtered_df["norm_neuropsych"] == 0)
]
superger_with_non_normal_neuropsych[
    [
        "id",
        "normSD_TMTB",
        "normSD_sf",
        "normSD_dsf",
        "normSD_TMTA",
        "normSD_dsb",
    ]
].head(50)

In [ ]:
# Create a new df 
clean_df = filtered_df

# Create a new variable with the superagers and those that meet the norm without dropping the others 
conditions = [
    (clean_df['superager'] == 1) & (clean_df['norm_neuropsych'] == 1),
    (clean_df['superager'] == 0) & (clean_df['norm_neuropsych'] == 1)
]

choices = [1, 0]

clean_df['sa_all'] = np.select(conditions, choices, default=2)

row_count = len(clean_df)
superager_count = clean_df["superager"].sum()
age_matched_controls = row_count - superager_count

print("Note that this is the number of participants who fit the classification for inclusion in the superager analysis at tp1 and have tp2 data available")
print(" ")
print(f"Number of participants: {row_count}")
print(f"Number of superagers: {superager_count:.0f}")
print(f"Number of age-matched controls: {age_matched_controls:.0f}")

# Get basic info about superagers
superager_df = clean_df[clean_df["sa_all"] == 1]

average_age = superager_df["age"].mean()
standard_deviation = superager_df["age"].std()

print(f"Average superager age: {average_age:.2f}")
print(f"Standard deviation: {standard_deviation:.2f}")

# Get basic info about controls
controls_df = clean_df[clean_df["sa_all"] == 0]
average_age = controls_df["age"].mean()
standard_deviation = controls_df["age"].std()

print(f"Average control age: {average_age:.2f}")
print(f"Standard deviation: {standard_deviation:.2f}")

clean_df[["id", "sa_all", "superager", "norm_neuropsych", "age", "YoE"]].head(15)

In [ ]:
# Export this df to a csv to use for future analysis

clean_df.to_csv(
    "/Users/rachelmorse/Documents/2023:2024/Data/BBHI/Exported data/superager_tp1.csv", index=False
)